In [5]:
import requests as rq
from bs4 import BeautifulSoup

url = "https://finance.naver.com/sise/sise_deposit.nhn"
data = rq.get(url)
data_html = BeautifulSoup(data.content)
parse_day = data_html.select_one(
    "div.subtop_sise_graph2 > ul.subtop_chart_note > li > span.tah"
).text

print(parse_day)

Cookie = ""

  |  2026.01.19


In [6]:
import re

biz_day = re.findall("[0-9]+", parse_day)
biz_day = "".join(biz_day)
print(biz_day)

20260119


In [7]:
# OTP를 받아오는 과정
from io import BytesIO

import pandas as pd
import requests as rq

gen_otp_url = "https://data.krx.co.kr/comm/fileDn/GenerateOTP/generate.cmd"
gen_otp_stk = {
    "mktId": "STK",
    "trdDd": biz_day,
    "money": "1",
    "csvxls_isNo": "false",
    "name": "fileDown",
    "url": "dbms/MDC/STAT/standard/MDCSTAT03901",
}

# User-Agent를 추가하여 실제 브라우저처럼 보이게 합니다.
headers = {
    "Referer": "https://data.krx.co.kr/contents/MDC/MDI/mdiLoader/index.cmd?menuId=MDC0201020203",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 \
    (KHTML, like Gecko) Version/18.6 Safari/605.1.15",
    "Cookie": Cookie,
}

otp_stk = rq.post(gen_otp_url, gen_otp_stk, headers=headers).text

print(otp_stk)

HDXDuwRT2eYe15H+LdVef5/7/B4a+EKfsBAmZX66WoYRtSksuLS7Bnxpl86F7dAOkunw9BBwugQaSjGAcH15eXNtMg1h8A3SskhjowOFFnUtBgM+EFJCxYg3zco1gIgRZqIo4cIzoURnTI8+MmkJ4m8vFLhSKmM794gFu+ThsO3xQSQdB+QHpETAf7Bzk0U2wGkKrPsfTuY8IIdJVkJu3HO5Fcc2uhj/l8Cz0y7OzNI=


In [8]:
down_url = "http://data.krx.co.kr/comm/fileDn/download_csv/download.cmd"
down_sector_stk = rq.post(down_url, {"code": otp_stk}, headers=headers)
sector_stk = pd.read_csv(BytesIO(down_sector_stk.content), encoding="euc-kr")

sector_stk.head()

,종목코드,종목명,시장구분,업종명,종가,대비,등락률,시가총액
0,095570,AJ네트웍스,KOSPI,일반서비스,4530,-30,-0.66,204994998270
1,006840,AK홀딩스,KOSPI,기타금융,7620,-280,-3.54,100946414820
2,027410,BGF,KOSPI,기타금융,3835,-10,-0.26,367073893485
3,282330,BGF리테일,KOSPI,유통,114400,-800,-0.69,1977278846400
4,138930,BNK금융지주,KOSPI,기타금융,14800,-140,-0.94,4592840088400


In [9]:
gen_otp_ksq = {
    "mktId": "KSQ",
    "trdDd": biz_day,
    "money": "1",
    "csvxls_isNo": "false",
    "name": "fileDown",
    "url": "dbms/MDC/STAT/standard/MDCSTAT03901",
}

otp_ksq = rq.post(gen_otp_url, gen_otp_ksq, headers=headers).text

down_sector_ksq = rq.post(down_url, {"code": otp_ksq}, headers=headers)
sector_ksq = pd.read_csv(BytesIO(down_sector_ksq.content), encoding="euc-kr")

sector_ksq.head()

,종목코드,종목명,시장구분,업종명,종가,대비,등락률,시가총액
0,060310,3S,KOSDAQ,의료·정밀기기,1505,-14,-0.92,79853855200
1,054620,APS,KOSDAQ,금융,4140,220,5.61,76152074940
2,265520,AP시스템,KOSDAQ,기계·장비,18880,50,0.27,284159500480
3,211270,AP위성,KOSDAQ,전기·전자,15090,520,3.57,227591967360
4,139050,BF랩스,KOSDAQ,IT 서비스,2805,0,0.00,24247965555


In [10]:
krx_sector = pd.concat([sector_stk, sector_ksq], axis=0, ignore_index=True).reset_index(drop=True)
krx_sector["종목명"] = krx_sector["종목명"].str.strip()
krx_sector["기준일"] = biz_day

krx_sector.head()

,종목코드,종목명,시장구분,업종명,종가,대비,등락률,시가총액,기준일
0,095570,AJ네트웍스,KOSPI,일반서비스,4530,-30,-0.66,204994998270,20260119
1,006840,AK홀딩스,KOSPI,기타금융,7620,-280,-3.54,100946414820,20260119
2,027410,BGF,KOSPI,기타금융,3835,-10,-0.26,367073893485,20260119
3,282330,BGF리테일,KOSPI,유통,114400,-800,-0.69,1977278846400,20260119
4,138930,BNK금융지주,KOSPI,기타금융,14800,-140,-0.94,4592840088400,20260119


In [11]:
gen_otp_url = "http://data.krx.co.kr/comm/fileDn/GenerateOTP/generate.cmd"

gen_top_data = {
    "searchType": "1",
    "mktId": "ALL",
    "trdDd": biz_day,
    "csvxls_isNo": "false",
    "name": "fileDown",
    "url": "dbms/MDC/STAT/standard/MDCSTAT03501",
}

headers = {
    "Referer": "https://data.krx.co.kr/contents/MDC/MDI/mdiLoader/index.cmd?menuId=MDC0201020203",
    "Cookie": Cookie,
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 \
    (KHTML, like Gecko) Version/18.6 Safari/605.1.15",
}

otp = rq.post(gen_otp_url, gen_top_data, headers=headers).text

In [12]:
down_url = "http://data.krx.co.kr/comm/fileDn/download_csv/download.cmd"
krx_ind = rq.post(down_url, {"code": otp}, headers=headers)
krx_ind = pd.read_csv(BytesIO(krx_ind.content), encoding="euc-kr")
krx_ind["종목명"] = krx_ind["종목명"].str.strip()
krx_ind["기준일"] = biz_day

krx_ind.head()

,종목코드,종목명,종가,대비,등락률,EPS,PER,선행 EPS,선행 PER,BPS,PBR,주당배당금,배당수익률,기준일
0,060310,3S,1505,-14,-0.92,NaN,NaN,NaN,NaN,933.0,1.61,0,0.00,20260119
1,095570,AJ네트웍스,4530,-30,-0.66,485.0,9.34,390.0,11.61,9902.0,0.46,270,5.96,20260119
2,006840,AK홀딩스,7620,-280,-3.54,NaN,NaN,NaN,NaN,40295.0,0.19,400,5.25,20260119
3,054620,APS,4140,220,5.61,NaN,NaN,NaN,NaN,11326.0,0.37,0,0.00,20260119
4,265520,AP시스템,18880,50,0.27,3448.0,5.48,NaN,NaN,22405.0,0.84,530,2.81,20260119


In [13]:
diff = list(set(krx_sector["종목명"]).symmetric_difference(set(krx_ind["종목명"])))
print(diff)

['코람코더원리츠', '에이리츠', '맵스리얼티', '로스웰', 'GRT', 'NH올원리츠', '이스트아시아홀딩스', '이지스레지던스리츠', '테라뷰', '크리스탈신소재', '신한알파리츠', '헝셩그룹', '케이탑리츠', '엘브이엠씨홀딩스', '씨엑스아이', '네오이뮨텍', '신한서부티엔디리츠', 'ESR켄달스퀘어리츠', '이지스밸류플러스리츠', '한화리츠', '코람코라이프인프라리츠', 'SK리츠', '대신밸류리츠', 'NH프라임리츠', '코오롱티슈진', 'KB스타리츠', '마스턴프리미어리츠', '롯데리츠', '제이알글로벌리츠', '애머릿지', '스타에스엠리츠', '잉글우드랩', 'KB발해인프라', '엑세스바이오', '윙입푸드', '컬러레이', '한국ANKOR유전', '삼성FN리츠', '미래에셋맵스리츠', '소마젠', '이리츠코크렙', '미래에셋글로벌리츠', '프레스티지바이오파마', '디앤디플랫폼리츠', '신한글로벌액티브리츠', '고스트스튜디오', '맥쿼리인프라', '오가닉티코스메틱', '글로벌에스엠', 'JTC']


In [14]:
kor_ticker = pd.merge(
    krx_sector, krx_ind, on=krx_sector.columns.intersection(krx_ind.columns).tolist(), how="outer"
)

kor_ticker.head()

,종목코드,종목명,시장구분,업종명,종가,대비,등락률,시가총액,기준일,EPS,PER,선행 EPS,선행 PER,BPS,PBR,주당배당금,배당수익률
0,000020,동화약품,KOSPI,제약,5950,-60,-1.00,166192246500,20260119,201.0,29.60,NaN,NaN,13470.0,0.44,180.0,3.03
1,000040,KR모터스,KOSPI,운송장비·부품,403,0,0.00,34809199152,20260119,NaN,NaN,NaN,NaN,543.0,0.74,0.0,0.00
2,000050,경방,KOSPI,유통,8280,80,0.98,226998435600,20260119,947.0,8.74,NaN,NaN,30489.0,0.27,150.0,1.81
3,000070,삼양홀딩스,KOSPI,기타금융,57700,-1000,-1.70,447936120700,20260119,3651.0,15.80,NaN,NaN,272260.0,0.21,3500.0,6.07
4,000080,하이트진로,KOSPI,음식료·담배,17920,-50,-0.28,1256794309120,20260119,1378.0,13.00,1651.0,10.85,16148.0,1.11,700.0,3.91


In [15]:
import numpy as np

kor_ticker["종목구분"] = np.where(
    kor_ticker["종목명"].str.contains("스팩|제[0-9]+호"),
    "스팩",
    np.where(
        kor_ticker["종목코드"].str[-1:] != "0",
        "우선주",
        np.where(
            kor_ticker["종목명"].str.endswith("리츠"),
            "리츠",
            np.where(kor_ticker["종목명"].isin(diff), "기타", "보통주"),
        ),
    ),
)

kor_ticker = kor_ticker.reset_index(drop=True)
kor_ticker.columns = kor_ticker.columns.str.replace(" ", "")
kor_ticker = kor_ticker[
    [
        "종목코드",
        "종목명",
        "시장구분",
        "종가",
        "시가총액",
        "기준일",
        "EPS",
        "선행EPS",
        "BPS",
        "주당배당금",
        "종목구분",
    ]
]

kor_ticker = kor_ticker.replace({np.nan: None})
kor_ticker["기준일"] = pd.to_datetime(kor_ticker["기준일"])

kor_ticker.head()

,종목코드,종목명,시장구분,종가,시가총액,기준일,EPS,선행EPS,BPS,주당배당금,종목구분
0,000020,동화약품,KOSPI,5950,166192246500,2026-01-19,201.0,None,13470.0,180.0,보통주
1,000040,KR모터스,KOSPI,403,34809199152,2026-01-19,None,None,543.0,0.0,보통주
2,000050,경방,KOSPI,8280,226998435600,2026-01-19,947.0,None,30489.0,150.0,보통주
3,000070,삼양홀딩스,KOSPI,57700,447936120700,2026-01-19,3651.0,None,272260.0,3500.0,보통주
4,000080,하이트진로,KOSPI,17920,1256794309120,2026-01-19,1378.0,1651.0,16148.0,700.0,보통주


In [18]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DB").getOrCreate()
# 간단한 데이터프레임 생성 및 확인
data = [("Samsung", 70000), ("SK Hynix", 180000)]
# df = spark.createDataFrame(data, ["Stock", "Price"])
# df.show()
df_spark = spark.createDataFrame(kor_ticker)
df_spark.show()

26/01/21 09:40:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+--------+------------------+--------+------+--------------+-------------------+-------+-------+--------+----------+--------+
|종목코드|            종목명|시장구분|  종가|      시가총액|             기준일|    EPS|선행EPS|     BPS|주당배당금|종목구분|
+--------+------------------+--------+------+--------------+-------------------+-------+-------+--------+----------+--------+
|  000020|          동화약품|   KOSPI|  5950|  166192246500|2026-01-19 00:00:00|  201.0|   NULL| 13470.0|     180.0|  보통주|
|  000040|          KR모터스|   KOSPI|   403|   34809199152|2026-01-19 00:00:00|   NULL|   NULL|   543.0|       0.0|  보통주|
|  000050|              경방|   KOSPI|  8280|  226998435600|2026-01-19 00:00:00|  947.0|   NULL| 30489.0|     150.0|  보통주|
|  000070|        삼양홀딩스|   KOSPI| 57700|  447936120700|2026-01-19 00:00:00| 3651.0|   NULL|272260.0|    3500.0|  보통주|
|  000080|        하이트진로|   KOSPI| 17920| 1256794309120|2026-01-19 00:00:00| 1378.0| 1651.0| 16148.0|     700.0|  보통주|
|  000087|    하이트진로2우B|   KOSPI| 14390|   16262685820|202

In [19]:
import pandas as pd
import requests as rq

url = f"""http://www.wiseindex.com/Index/GetIndexComponets?ceil_yn=0&dt={biz_day}&sec_cd=G10"""
data = rq.get(url).json()

type(data)

dict

In [20]:
print(data.keys())

dict_keys(['info', 'list', 'sector', 'size'])


In [21]:
data["list"][0]

{'IDX_CD': 'G10',
 'IDX_NM_KOR': 'WICS 에너지',
 'ALL_MKT_VAL': 32384186,
 'CMP_CD': '034730',
 'CMP_KOR': 'SK',
 'MKT_VAL': 9805265,
 'WGT': 30.28,
 'S_WGT': 30.28,
 'CAL_WGT': 1.0,
 'SEC_CD': 'G10',
 'SEC_NM_KOR': '에너지',
 'SEQ': 1,
 'TOP60': 3,
 'APT_SHR_CNT': 33351243}

In [22]:
data["sector"]

[{'SEC_CD': 'G25', 'SEC_NM_KOR': '경기관련소비재', 'SEC_RATE': 8.2, 'IDX_RATE': 0},
 {'SEC_CD': 'G35', 'SEC_NM_KOR': '건강관리', 'SEC_RATE': 8.46, 'IDX_RATE': 0},
 {'SEC_CD': 'G50', 'SEC_NM_KOR': '커뮤니케이션서비스', 'SEC_RATE': 3.75, 'IDX_RATE': 0},
 {'SEC_CD': 'G40', 'SEC_NM_KOR': '금융', 'SEC_RATE': 8.08, 'IDX_RATE': 0},
 {'SEC_CD': 'G10', 'SEC_NM_KOR': '에너지', 'SEC_RATE': 1.62, 'IDX_RATE': 100.0},
 {'SEC_CD': 'G20', 'SEC_NM_KOR': '산업재', 'SEC_RATE': 20.19, 'IDX_RATE': 0},
 {'SEC_CD': 'G55', 'SEC_NM_KOR': '유틸리티', 'SEC_RATE': 1.21, 'IDX_RATE': 0},
 {'SEC_CD': 'G30', 'SEC_NM_KOR': '필수소비재', 'SEC_RATE': 1.43, 'IDX_RATE': 0},
 {'SEC_CD': 'G15', 'SEC_NM_KOR': '소재', 'SEC_RATE': 4.36, 'IDX_RATE': 0},
 {'SEC_CD': 'G45', 'SEC_NM_KOR': 'IT', 'SEC_RATE': 42.69, 'IDX_RATE': 0}]

In [23]:
data_pd = pd.json_normalize(data["list"])

data_pd

,IDX_CD,IDX_NM_KOR,ALL_MKT_VAL,CMP_CD,CMP_KOR,MKT_VAL,WGT,S_WGT,CAL_WGT,SEC_CD,SEC_NM_KOR,SEQ,TOP60,APT_SHR_CNT
0,G10,WICS 에너지,32384186,034730,SK,9805265,30.28,30.28,1.0,G10,에너지,1,3,33351243
1,G10,WICS 에너지,32384186,096770,SK이노베이션,6822294,21.07,51.34,1.0,G10,에너지,2,3,64240059
2,G10,WICS 에너지,32384186,010950,S-Oil,3753173,11.59,62.93,1.0,G10,에너지,3,3,41655633
3,G10,WICS 에너지,32384186,009830,한화솔루션,3043014,9.40,72.33,1.0,G10,에너지,4,3,108292297
4,G10,WICS 에너지,32384186,078930,GS,2671503,8.25,80.58,1.0,G10,에너지,5,3,44599381
5,G10,WICS 에너지,32384186,083650,비에이치아이,1117339,3.45,84.03,1.0,G10,에너지,6,3,18257181
6,G10,WICS 에너지,32384186,112610,씨에스윈드,945820,2.92,86.95,1.0,G10,에너지,7,3,23615985
7,G10,WICS 에너지,32384186,100090,SK오션플랜트,774729,2.39,89.34,1.0,G10,에너지,8,3,38736439
8,G10,WICS 에너지,32384186,006120,SK디스커버리,461595,1.43,90.77,1.0,G10,에너지,9,3,7706099
9,G10,WICS 에너지,32384186,475150,SK이터닉스,375588,1.16,91.93,1.0,G10,에너지,10,3,18902263


In [24]:
import time

import pandas as pd
import requests as rq
from tqdm import tqdm

sector_code = ["G25", "G35", "G50", "G40", "G10", "G20", "G55", "G30", "G15", "G45"]

data_sector = []

for i in tqdm(sector_code):
    url = f"""http://www.wiseindex.com/Index/GetIndexComponets?ceil_yn=0&dt={biz_day}&sec_cd={i}"""
    data = rq.get(url).json()
    data_sector.append(pd.json_normalize(data["list"]))

    time.sleep(2)

kor_sector = pd.concat(data_sector, axis=0)
kor_sector = kor_sector[["IDX_CD", "CMP_CD", "CMP_KOR", "SEC_NM_KOR"]]
kor_sector["기준일"] = biz_day
kor_sector["기준일"] = pd.to_datetime(kor_sector["기준일"])

100%|██████████| 10/10 [00:23<00:00,  2.37s/it]


In [25]:
df_spark.show()

+--------+------------------+--------+------+--------------+-------------------+-------+-------+--------+----------+--------+
|종목코드|            종목명|시장구분|  종가|      시가총액|             기준일|    EPS|선행EPS|     BPS|주당배당금|종목구분|
+--------+------------------+--------+------+--------------+-------------------+-------+-------+--------+----------+--------+
|  000020|          동화약품|   KOSPI|  5950|  166192246500|2026-01-19 00:00:00|  201.0|   NULL| 13470.0|     180.0|  보통주|
|  000040|          KR모터스|   KOSPI|   403|   34809199152|2026-01-19 00:00:00|   NULL|   NULL|   543.0|       0.0|  보통주|
|  000050|              경방|   KOSPI|  8280|  226998435600|2026-01-19 00:00:00|  947.0|   NULL| 30489.0|     150.0|  보통주|
|  000070|        삼양홀딩스|   KOSPI| 57700|  447936120700|2026-01-19 00:00:00| 3651.0|   NULL|272260.0|    3500.0|  보통주|
|  000080|        하이트진로|   KOSPI| 17920| 1256794309120|2026-01-19 00:00:00| 1378.0| 1651.0| 16148.0|     700.0|  보통주|
|  000087|    하이트진로2우B|   KOSPI| 14390|   16262685820|202

DataFrame[종목코드: string, 종목명: string, 시장구분: string, 종가: bigint, 시가총액: bigint, 기준일: timestamp, EPS: double, 선행EPS: double, BPS: double, 주당배당금: double, 종목구분: string]

26/01/21 04:53:32 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1062600 ms exceeds timeout 120000 ms
26/01/21 04:53:32 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/21 05:11:26 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at 

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("stock_db").getOrCreate()

In [3]:
import keyring

keyring.set_password("tiingo", "user_name", "api_key")

In [8]:
import keyring
import pandas as pd
from tiingo import TiingoClient

api_key = keyring.get_password("tiingo", "user_name")
config = {}
config["session"] = True
config["api_key"] = api_key
client = TiingoClient(config=config)

In [10]:
tickers = client.list_stock_tickers()
tickers_df = pd.DataFrame.from_records(tickers)

tickers_df.head()

,ticker,exchange,assetType,priceCurrency,startDate,endDate
0,-P-H,NYSE,Stock,USD,,
1,-P-HIZ,NASDAQ,Stock,USD,2023-08-30,2025-09-29
2,-P-S,NYSE,Stock,USD,2018-08-22,2023-05-05
3,000001,SHE,Stock,CNY,2007-01-04,2026-01-23
4,000002,SHE,Stock,CNY,2007-01-04,2026-01-23


In [11]:
tickers_df.groupby(["exchange", "priceCurrency"])["ticker"].count()

exchange   priceCurrency
           USD               2447
AMEX       USD                 99
ASX        AUD                163
           USD               2111
BATS       USD                 85
CSE        USD                 32
EXPM       USD               3306
LSE        USD                 12
NASDAQ     USD              13441
NMFQS      USD                 36
NYSE       USD               7970
NYSE ARCA  USD                183
NYSE MKT   USD                481
NYSE NAT   USD                  3
OTCBB      USD                643
OTCCE      USD               1093
OTCD       USD                523
OTCGREY    USD               4462
OTCMKTS    USD               1181
OTCQB      USD               1319
OTCQX      USD                821
PINK       USD              14983
SHE        CNY               3386
           HKD                 12
SHEB       HKD                 42
SHG        CNY               2954
           USD                  6
SHGB       USD                 44
Name: ticker, dtype: in

In [12]:
ticker_metadata = client.get_ticker_metadata("AAPL")
print(ticker_metadata)

{'ticker': 'AAPL', 'name': 'Apple Inc', 'description': "Apple Inc. (Apple) designs, manufactures and markets mobile communication and media devices, personal computers, and portable digital music players, and a variety of related software, services, peripherals, networking solutions, and third-party digital content and applications. The Company's products and services include iPhone, iPad, Mac, iPod, Apple TV, a portfolio of consumer and professional software applications, the iOS and OS X operating systems, iCloud, and a variety of accessory, service and support offerings. The Company also delivers digital content and applications through the iTunes Store, App StoreSM, iBookstoreSM, and Mac App Store. The Company distributes its products worldwide through its retail stores, online stores, and direct sales force, as well as through third-party cellular network carriers, wholesalers, retailers, and value-added resellers. In February 2012, the Company acquired app-search engine Chomp.", 

In [13]:
historical_prices = client.get_dataframe("AAPL", startDate="2017-08-01", frequency="daily")

historical_prices.head()

,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor
date,,,,,,,,,,,,
2017-08-01 00:00:00+00:00,150.05,150.22,148.4100,149.10,24725526,34.874739,34.914251,34.493569,34.653940,98902104,0.0,1.0
2017-08-02 00:00:00+00:00,157.14,159.75,156.1600,159.28,69222793,36.522603,37.129221,36.294830,37.019983,276891172,0.0,1.0
2017-08-03 00:00:00+00:00,155.57,157.21,155.0200,157.05,26000738,36.157702,36.538872,36.029871,36.501685,104002952,0.0,1.0
2017-08-04 00:00:00+00:00,156.39,157.40,155.6900,156.07,20349532,36.348287,36.583032,36.185593,36.273912,81398128,0.0,1.0
2017-08-07 00:00:00+00:00,158.81,158.92,156.6701,157.06,21870321,36.910745,36.936312,36.413388,36.504009,87481284,0.0,1.0


In [14]:
fundamentals_daily = client.get_fundamentals_daily("AAPL")
fundamentals_daily_df = pd.DataFrame.from_records(fundamentals_daily)

fundamentals_daily_df.head()

,date,marketCap,enterpriseVal,peRatio,pbRatio,trailingPEG1Y
0,2023-01-25T00:00:00.000Z,2.256726e+12,2.316481e+12,23.712324,39.782213,-2.274227
1,2023-01-26T00:00:00.000Z,2.290133e+12,2.349888e+12,24.063346,40.371123,-2.307894
2,2023-01-27T00:00:00.000Z,2.321472e+12,2.381227e+12,24.392637,40.923575,-2.339476
3,2023-01-30T00:00:00.000Z,2.274861e+12,2.334616e+12,23.902879,40.101907,-2.292503
4,2023-01-31T00:00:00.000Z,2.295382e+12,2.355137e+12,24.118506,40.463665,-2.313184
